In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import pandas as pd
import zipfile
import os
import time
import gc
from PIL import Image
from tqdm.notebook import tqdm
from utils import MetricsEngine, visual_compare
import torch.nn.functional as F

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Using device: cuda
Initial VRAM Allocated: 443.64 MB


In [ ]:
def get_positional_encoding(coords, num_frequencies=15):
    """Pre-computes positional encoding for the entire grid to save compute during training."""
    encoded = [coords]
    for i in range(num_frequencies):
        for fn in [torch.sin, torch.cos]:
            encoded.append(fn((2.0 ** i) * torch.pi * coords))
    return torch.cat(encoded, dim=-1)

def load_and_prep_data(image_path, downscale_factor=4, pos_enc_freqs=15):
    """Loads image, generates coords, pre-computes PE, and locks everything to GPU."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    new_w, new_h = w // downscale_factor, h // downscale_factor
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    img_np = np.array(img)
    img_norm = img_np / 255.0
    
    #Create Raw Coordinates
    y_coords = np.linspace(-1, 1, new_h)
    x_coords = np.linspace(-1, 1, new_w)
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    coords_raw = torch.tensor(np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1), dtype=torch.float32)
    
    #Pre-compute Positional Encodings
    coords_encoded = get_positional_encoding(coords_raw, pos_enc_freqs)
    
    #Colors
    colors = torch.tensor(img_norm.reshape(-1, 3), dtype=torch.float32)
    
    #Push EVERYTHING to GPU permanently
    return {
        "coords_raw": coords_raw.to(device),
        "coords_encoded": coords_encoded.to(device),
        "colors": colors.to(device),
        "h": new_h, "w": new_w,
        "num_pixels": new_h * new_w,
        "original_np": img_np
    }

#Load Data
dataset = load_and_prep_data("INSERT IMAGE HERE.jpg", downscale_factor=4, pos_enc_freqs=15)
print(f"Resolution: {dataset['w']}x{dataset['h']} | Total Pixels: {dataset['num_pixels']}")
print(f"VRAM after data load: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Resolution: 1404x936 | Total Pixels: 1314144
VRAM after data load: 779.52 MB


In [19]:
#STANDARD ReLU MLP
class ReLUMlp(nn.Module):
    def __init__(self, hidden_dim, num_layers, pos_enc_freqs):
        super().__init__()
        input_dim = 2 + 4 * pos_enc_freqs 
        
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
            
        layers.extend([nn.Linear(hidden_dim, 3), nn.Sigmoid()])
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

#PURE SIREN
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features)
        self.init_weights()
        
    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                # SIREN Paper: First layer uniform (-1/in, 1/in)
                self.linear.weight.uniform_(-1 / self.linear.in_features, 1 / self.linear.in_features)
            else:
                # SIREN Paper: Hidden layers uniform (-sqrt(6/in)/w0, sqrt(6/in)/w0)
                bound = np.sqrt(6 / self.linear.in_features) / self.omega_0
                self.linear.weight.uniform_(-bound, bound)
                
    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

class SirenMLP(nn.Module):
    def __init__(self, hidden_dim, num_layers, omega_0=30.0):
        super().__init__()
        self.net = nn.Sequential()
        
        #First Layer (Receives raw 2D coords)
        self.net.append(SineLayer(2, hidden_dim, is_first=True, omega_0=omega_0))
        
        #Hidden Layers
        for _ in range(num_layers - 2):
            self.net.append(SineLayer(hidden_dim, hidden_dim, is_first=False, omega_0=omega_0))
            
        #Output Layer (To RGB)
        final_linear = nn.Linear(hidden_dim, 3)
        with torch.no_grad():
            # Standard init for final layer
            bound = np.sqrt(6 / hidden_dim) / omega_0
            final_linear.weight.uniform_(-bound, bound)
            
        self.net.append(final_linear)
        self.net.append(nn.Sigmoid())
        
    def forward(self, x):
        return self.net(x)


In [20]:
def train_and_evaluate(config, dataset):
    model_name = config["name"]
    print(f"\n=== Starting: {model_name} ===")
    
    # 1. Architecture Selection
    if config["activation"] == "sine":
        model = SirenMLP(config["dim"], config["layers"]).to(device)
        input_data = dataset["coords_raw"]
    else:
        model = ReLUMlp(config["dim"], config["layers"], config["freqs"]).to(device)
        input_data = dataset["coords_encoded"]
        
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    #Cosine Annealing Scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-6)
    criterion = nn.MSELoss()
    
    scaler = torch.amp.GradScaler('cuda')
    
    batch_size = config["batch_size"]
    num_pixels = dataset["num_pixels"]
    
    model.train()
    pbar = tqdm(range(config["epochs"]), desc=model_name)
    
    for epoch in pbar:

        indices = torch.randperm(num_pixels, device=device)
        
        epoch_loss = 0.0
        batches = 0
        

        for i in range(0, num_pixels, batch_size):
            batch_idx = indices[i : i + batch_size]
            b_coords = input_data[batch_idx]
            b_colors = dataset["colors"][batch_idx]
            
            optimizer.zero_grad(set_to_none=True) 
            
            #AMP Forward Pass
            with torch.amp.autocast('cuda'):
                pred = model(b_coords)
                loss = criterion(pred, b_colors)
                
            #AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.detach()
            batches += 1
            
        scheduler.step()
        
        #Update UI every 50 epochs
        if epoch % 50 == 0:
            avg_loss = epoch_loss / batches
            psnr = -10.0 * torch.log10(avg_loss).item()
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({"PSNR": f"{psnr:.2f}", "LR": f"{current_lr:.1e}"})

    #EVAL STAGE
    model.eval()
    start_time = time.time()
    predicted_colors = []
    
    #Chunked Inference with AMP
    with torch.no_grad(), torch.amp.autocast('cuda'):
        chunk_size = 65536 
        for i in range(0, num_pixels, chunk_size):
            chunk = input_data[i : i + chunk_size]
            predicted_colors.append(model(chunk))
            
        full_pred = torch.cat(predicted_colors, dim=0)
        
    torch.cuda.synchronize()
    latency_ms = (time.time() - start_time) * 1000

    #Save Output Image
    pred_np = full_pred.float().cpu().numpy().reshape(dataset["h"], dataset["w"], 3)
    pred_img_uint8 = (pred_np * 255).clip(0, 255).astype(np.uint8)
    os.makedirs("images", exist_ok=True)
    cv2.imwrite(f"images/{model_name}.png", cv2.cvtColor(pred_img_uint8, cv2.COLOR_RGB2BGR))

    #EXTREME STORAGE MINIMIZATION
    os.makedirs("models", exist_ok=True)
    pth_path = f"models/{model_name}.pth"
    zip_path = f"models/{model_name}.zip"
    
    # Convert weights to FP16 before saving (Halves the size immediately)
    model.half()
    torch.save(model.state_dict(), pth_path)
    
    # Compress using LZMA
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_LZMA) as zipf:
        zipf.write(pth_path, arcname=f"{model_name}.pth")
    size_kb = os.path.getsize(zip_path) / 1024.0

    # Compute Metrics
    engine = MetricsEngine(device)
    metrics = engine.compute_all(dataset["original_np"], pred_img_uint8)
    
    row = {"Method": model_name, "Size_KB": round(size_kb, 2), "Latency_ms": round(latency_ms, 2)}
    row.update(metrics)
    
    #CLEAN SLATE(VRAM PURGE)
    del model
    del optimizer
    del scheduler
    del scaler
    del full_pred
    del predicted_colors
    del criterion
    gc.collect()
    torch.cuda.empty_cache()
    
    return row

In [ ]:
batchsize = dataset["num_pixels"]

EXPERIMENTS = [ 
#{"name":"ReLU_Baseline", "activation": "relu", "layers": 5, "dim": 256, "freqs": 15,"epochs": 400, "batch_size": 131072, "lr": 1e-3},
#{"name":"SIREN_Tiny", "activation": "sine", "layers": 6, "dim": 64, "freqs": 16,"epochs": 500, "batch_size": 65536, "lr": 1e-3},
#{"name":"SIREN_Medium", "activation": "sine", "layers": 4, "dim": 256, "freqs": 10,"epochs": 300, "batch_size": 65536, "lr": 1e-3},
#{"name":"SIREN_Large", "activation": "sine", "layers": 5, "dim": 256, "freqs": 15,"epochs": 400, "batch_size": 65536, "lr": 1e-3}
]

os.makedirs("results", exist_ok=True) 
all_neural_results = []

for config in EXPERIMENTS: 
    result = train_and_evaluate(config, dataset)
    all_neural_results.append(result)
    torch.cuda.empty_cache()

#save intermediate in case of crash
pd.DataFrame(all_neural_results).to_csv("results/RELUandSIRENtiny_metrics_temp.csv", index=False)
print(f"Finished {config['name']} | VRAM Reset to: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

#Final Save
df_neural = pd.DataFrame(all_neural_results)
df_neural.to_csv("results/RELUandSIRENtiny_metrics.csv", index=False) 
print("\nAllexperiments completed and fully optimized!") 
display(df_neural)


=== Starting: SIREN_Tiny ===


SIREN_Tiny:   0%|          | 0/500 [00:00<?, ?it/s]

Finished SIREN_Tiny | VRAM Reset to: 421.23 MB

Allexperiments completed and fully optimized!


,Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
0,SIREN_Tiny,31.77,24.71,27.587898,0.770252,0.983445,0.503359
